In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DataCleaningDemo") \
    .master("local[*]") \
    .getOrCreate()
print("SparkSession created successfully.")

In [ ]:
data = [
    (101, "  Amit  ", 25, "Pune", 30000),
    (102, "NEHA", None, "Mumbai", 35000),
    (103, "rahul", 28, None, 40000),
    (104, "Priya", 32, "pune", None),
    (105, "Rohan gupta", 27, "MUMBAI", 28000),
    (105, "Rohan", 27, "MUMBAI", 28000)
]

columns = ["EmployeeID", "Name", "Age", "City", "Salary"]

In [ ]:
df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
# Count null values in each column
from pyspark.sql.functions import col, sum as spark_sum

df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [ ]:
# Filter rows where Age is null
df.filter(col("Age").isNull()).show()
df.filter(col("Age").isNotNull()).show()

In [ ]:
# Drop rows with any null values
clean_df = df.na.drop()
clean_df.show()

In [ ]:
# Drop rows where Age or Salary is null
clean_df = df.na.drop(
    subset=["Age", "Salary"]
)

clean_df.show()

In [ ]:
# Fill null values with a specific value
df2 = df.na.fill({
    "Age": 0
})

df2.show()

In [ ]:
# Fill null values in the "City" column with "Unknown"
df2 = df.na.fill({
    "City": "Unknown"
})

df2.show()

In [ ]:
# Fill null values in the "Age" column with the average age
from pyspark.sql.functions import avg

average_age = df.select(
    avg("Age")
).first()[0]

print("Average Age:", average_age)

df2 = df.na.fill({
    "Age": int(average_age)
})

df2.show()

In [ ]:
# Identify duplicate rows based on all columns
duplicate_rows = df.groupBy(
    df.columns
).count().filter(
    col("count") > 1
)

duplicate_rows.show()

In [ ]:
# Identify duplicate rows based on a specific column (e.g., EmployeeID)
df.groupBy("EmployeeID") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [ ]:
# Remove duplicate rows based on all columns
clean_df = df.distinct()
clean_df.show()

In [ ]:
# Remove duplicate rows based on all columns
clean_df = df.dropDuplicates()
clean_df.show()

In [ ]:
# Remove duplicate rows based on a specific column (e.g., EmployeeID)
clean_df = df.dropDuplicates(
    ["EmployeeID"]
)

clean_df.show()

In [ ]:
# Remove leading and trailing spaces from the "Name" column
from pyspark.sql.functions import trim

df2 = df.withColumn(
    "Cleaned_Name",
    trim(col("Name"))
)

df2.show()

In [ ]:
# Convert the "Name" column to lowercase
from pyspark.sql.functions import lower

df2 = df.withColumn(
    "Name_In_LowerCase",
    lower(trim(col("Name")))
)

df2.show()

In [ ]:
# Convert the "City" column to uppercase
from pyspark.sql.functions import upper

df2 = df.withColumn(
    "City",
    upper(col("City"))
)

df2.show()

In [ ]:
# Convert the "Name" column to title case (capitalize the first letter of each word)
from pyspark.sql.functions import initcap

df2 = df.withColumn(
    "Name",
    initcap(trim(col("Name")))
)

df2.show()

In [ ]:
# Remove special characters from the "Phone" column
phone_data = [
    ("Amit", "98765-43210"),
    ("Neha", "98765 12345"),
    ("Rahul", "9876543210")
]

phone_df = spark.createDataFrame(
    phone_data,
    ["Name", "Phone"]
)

from pyspark.sql.functions import regexp_replace

phone_df = phone_df.withColumn(
    "Phone",
    regexp_replace(
        col("Phone"),
        "[- ]",
        ""
    )
)

phone_df.show()

In [ ]:
name_data = [
    ("Amit", "Sharma"),
    ("Neha", "Patel"),
    ("Rahul", "Verma")
]

name_df = spark.createDataFrame(
    name_data,
    ["FirstName", "LastName"]
)

from pyspark.sql.functions import concat, lit

# Create a new column "FullName" by concatenating "FirstName" and "LastName"
name_df = name_df.withColumn(
    "FullName",
    concat(
        col("FirstName"),
        lit(" "),
        col("LastName")
    )
)

name_df.show()

In [ ]:
location_data = [
    ("Amit", "Pune,Maharashtra"),
    ("Neha", "Mumbai,Maharashtra"),
    ("Rahul", "Ahmedabad,Gujarat")
]

location_df = spark.createDataFrame(
    location_data,
    ["Name", "Location"]
)


from pyspark.sql.functions import split

# Split the "Location" column into "City" and "State"
location_df = location_df \
    .withColumn(
        "City",
        split(col("Location"), ",")[0]
    ) \
    .withColumn(
        "State",
        split(col("Location"), ",")[1]
    )

location_df.show()

In [ ]:
# Identify rows with invalid age values (e.g., less than 18 or greater than 60)
invalid_age = df.filter(
    (col("Age") < 18) |
    (col("Age") > 60)
)

invalid_age.show()

In [ ]:
# Identify rows with invalid email addresses (e.g., missing "@" or ".")
email_data = [
    ("Amit", "amit@gmail.com"),
    ("Neha", "neha@gmail.com"),
    ("Rahul", "rahul@"),
    ("Priya", "priyagmail.com")
]

email_df = spark.createDataFrame(
    email_data,
    ["Name", "Email"]
)

email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

invalid_emails = email_df.filter(
    ~col("Email").rlike(email_pattern)
)

invalid_emails.show()

In [ ]:
salary_data = [
    ("Amit", "30000"),
    ("Neha", "35000"),
    ("Rahul", "40000")
]

salary_df = spark.createDataFrame(
    salary_data,
    ["Name", "Salary"]
)

salary_df.printSchema()

salary_df = salary_df.withColumn(
    "Salary",
    col("Salary").cast("integer")
)

salary_df.printSchema()